# 03 - PDF Chunking Hands-On (Profile-Based)

This notebook processes all PDFs inside:
- `Documents/PDF Dummy Documents/AI Interview PDFs`
- `Documents/PDF Dummy Documents/Different Types Example`

and creates high-quality chunk artifacts in:
- `Handson/01-Chunking/data/pdf/ai_interview_pdfs`
- `Handson/01-Chunking/data/pdf/different_types_example`
- `Handson/01-Chunking/data/pdf/quality_reports`

Design principle used here:
- **AI interview PDFs**: compact scoring/report structure, use moderate chunk size and strong metadata.
- **Different types PDFs**: mixed content (invoice/resume/scanned), use robust cleaning + adaptive handling + OCR-needed flag when no text layer exists.


In [ ]:
# Cell 1: Imports

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re

import fitz  # PyMuPDF
import pandas as pd
import tiktoken

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

enc = tiktoken.get_encoding('cl100k_base')

print('fitz / pymupdf ready')
print('pandas version  :', pd.__version__)
print('tiktoken version:', tiktoken.__version__)


In [ ]:
# Cell 2: Paths (adjust only if your repo root path is different)

REPO_ROOT = Path(r"C:/Users/Lenovo/source/repos/RAG_Handson")
PDF_ROOT = REPO_ROOT / "Documents" / "PDF Dummy Documents"

AI_PDF_DIR = PDF_ROOT / "AI Interview PDFs"
MIXED_PDF_DIR = PDF_ROOT / "Different Types Example"

OUT_ROOT = REPO_ROOT / "Handson" / "01-Chunking" / "data" / "pdf"
OUT_AI = OUT_ROOT / "ai_interview_pdfs"
OUT_MIXED = OUT_ROOT / "different_types_example"
OUT_REPORTS = OUT_ROOT / "quality_reports"

for p in [OUT_ROOT, OUT_AI, OUT_MIXED, OUT_REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print('AI PDF dir      :', AI_PDF_DIR)
print('Mixed PDF dir   :', MIXED_PDF_DIR)
print('Output root     :', OUT_ROOT)

assert AI_PDF_DIR.exists(), f'Missing: {AI_PDF_DIR}'
assert MIXED_PDF_DIR.exists(), f'Missing: {MIXED_PDF_DIR}'


In [ ]:
# Cell 3: Discover PDFs

ai_pdf_files = sorted(AI_PDF_DIR.glob('*.pdf'))
mixed_pdf_files = sorted(MIXED_PDF_DIR.glob('*.pdf'))
all_pdf_files = ai_pdf_files + mixed_pdf_files

print(f'AI Interview PDFs         : {len(ai_pdf_files)}')
for p in ai_pdf_files:
    print('  -', p.name)

print(f'\nDifferent Types PDFs      : {len(mixed_pdf_files)}')
for p in mixed_pdf_files:
    print('  -', p.name)

print(f'\nTotal PDFs                : {len(all_pdf_files)}')


In [ ]:
# Cell 4: Utility methods

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def stable_id(text: str, prefix: str = 'chunk') -> str:
    h = hashlib.md5(text.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}_{h}'

def clean_pdf_text(text: str) -> str:
    # Fix hyphenated line breaks
    text = re.sub(r'-\n', '', text)
    # Collapse excessive new lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove common page-number patterns
    text = re.sub(r'Page\s*\d+\s*(of|/)\s*\d+', '', text, flags=re.IGNORECASE)
    # Normalize ligatures
    text = text.replace('ﬁ', 'fi').replace('ﬂ', 'fl')
    # Remove null bytes
    text = text.replace('\x00', '')
    # Normalize whitespace
    text = ' '.join(text.split())
    return text.strip()

def pick_pdf_profile(pdf_path: Path) -> str:
    folder = pdf_path.parent.name.lower()
    if 'ai interview' in folder:
        return 'ai_interview'
    return 'mixed_pdf'


In [ ]:
# Cell 5: Profile config (best-fit defaults for this corpus)

PROFILE_CONFIG = {
    'ai_interview': {
        'chunk_size': 850,
        'chunk_overlap': 140,
        'separators': ['\n\n', '\n', '. ', ' ', ''],
        'doc_type': 'pdf_ai_interview',
    },
    'mixed_pdf': {
        'chunk_size': 900,
        'chunk_overlap': 120,
        'separators': ['\n\n', '\n', '. ', ' ', ''],
        'doc_type': 'pdf_mixed',
    },
}

PROFILE_CONFIG


In [ ]:
# Cell 6: Extract pages and basic quality stats

def extract_pdf_pages(pdf_path: Path) -> tuple[list[dict], dict]:
    doc = fitz.open(pdf_path)
    pages = []
    text_chars_per_page = []

    for i, page in enumerate(doc):
        raw = page.get_text('text') or ''
        cleaned = clean_pdf_text(raw)
        text_chars_per_page.append(len(cleaned))
        pages.append({
            'page_number': i + 1,
            'raw_text': raw,
            'cleaned_text': cleaned,
        })

    doc.close()

    nonempty_pages = sum(1 for x in text_chars_per_page if x > 0)
    summary = {
        'source_file': str(pdf_path),
        'file_name': pdf_path.name,
        'total_pages': len(pages),
        'nonempty_pages': nonempty_pages,
        'avg_cleaned_chars_nonempty': round(sum(x for x in text_chars_per_page if x > 0) / nonempty_pages, 1) if nonempty_pages else 0,
        'max_cleaned_chars_page': max(text_chars_per_page) if text_chars_per_page else 0,
        'ocr_needed': nonempty_pages == 0,
    }
    return pages, summary

profile_rows = []
for pdf in all_pdf_files:
    _, s = extract_pdf_pages(pdf)
    s['profile'] = pick_pdf_profile(pdf)
    profile_rows.append(s)

profile_df = pd.DataFrame(profile_rows)
display(profile_df[['file_name', 'profile', 'total_pages', 'nonempty_pages', 'avg_cleaned_chars_nonempty', 'ocr_needed']])


In [ ]:
# Cell 7: Chunking method (per file)

def chunk_single_pdf(pdf_path: Path) -> tuple[list[Document], dict]:
    profile = pick_pdf_profile(pdf_path)
    cfg = PROFILE_CONFIG[profile]

    pages, summary = extract_pdf_pages(pdf_path)

    # If no text-layer exists, return empty chunks and flag OCR requirement.
    if summary['ocr_needed']:
        summary.update({
            'chunks_created': 0,
            'avg_chunk_tokens': 0,
            'max_chunk_tokens': 0,
        })
        return [], summary

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
        separators=cfg['separators'],
    )

    chunks: list[Document] = []
    for p in pages:
        text = p['cleaned_text']
        if len(text.strip()) < 40:
            continue

        page_docs = splitter.create_documents(
            texts=[text],
            metadatas=[{
                'source': str(pdf_path),
                'file_name': pdf_path.name,
                'page': p['page_number'],
                'total_pages': summary['total_pages'],
                'doc_type': cfg['doc_type'],
                'profile': profile,
                'ocr_needed': False,
                'extraction_method': 'pymupdf_text_layer',
            }]
        )
        chunks.extend(page_docs)

    # Enrich metadata
    token_counts = []
    for i, c in enumerate(chunks):
        c.metadata['chunk_index'] = i
        c.metadata['chunk_id'] = stable_id(c.page_content, prefix='pdf')
        c.metadata['char_count'] = len(c.page_content)
        c.metadata['token_count'] = count_tokens(c.page_content)
        token_counts.append(c.metadata['token_count'])

    summary.update({
        'chunks_created': len(chunks),
        'avg_chunk_tokens': round(sum(token_counts) / len(token_counts), 1) if token_counts else 0,
        'max_chunk_tokens': max(token_counts) if token_counts else 0,
    })

    return chunks, summary

# Dry-run one sample from each folder
sample_files = [ai_pdf_files[0], mixed_pdf_files[0]]
for sf in sample_files:
    ch, sm = chunk_single_pdf(sf)
    print('\n' + '=' * 90)
    print('File:', sf.name)
    print('Profile:', sm['profile'])
    print('Chunks:', sm['chunks_created'])
    print('OCR needed:', sm['ocr_needed'])
    if ch:
        print('Preview metadata:', ch[0].metadata)
        print('Preview text:', ch[0].page_content[:250])


In [ ]:
# Cell 8: Batch chunk all PDFs

all_chunks: list[Document] = []
quality_rows: list[dict] = []

for pdf in all_pdf_files:
    chunks, summary = chunk_single_pdf(pdf)
    all_chunks.extend(chunks)
    quality_rows.append(summary)

quality_df = pd.DataFrame(quality_rows)
display(quality_df[['file_name', 'profile', 'total_pages', 'nonempty_pages', 'chunks_created', 'avg_chunk_tokens', 'max_chunk_tokens', 'ocr_needed']])

print('Total chunks created:', len(all_chunks))
print('Files needing OCR:', int(quality_df['ocr_needed'].sum()))


In [ ]:
# Cell 9: Save per-file JSONL chunks and consolidated folder artifacts

def save_jsonl(chunks: list[Document], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', encoding='utf-8') as f:
        for c in chunks:
            rec = {'page_content': c.page_content, 'metadata': c.metadata}
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

def split_chunks_by_source(chunks: list[Document]) -> dict[str, list[Document]]:
    by_source: dict[str, list[Document]] = {}
    for c in chunks:
        src = c.metadata['source']
        by_source.setdefault(src, []).append(c)
    return by_source

chunks_by_source = split_chunks_by_source(all_chunks)

# Save per-file artifacts
for src, ch in chunks_by_source.items():
    src_path = Path(src)
    profile = pick_pdf_profile(src_path)
    folder = OUT_AI if profile == 'ai_interview' else OUT_MIXED
    out_file = folder / f"{src_path.stem}_chunks.jsonl"
    save_jsonl(ch, out_file)

# Save consolidated artifacts per folder
ai_chunks = [c for c in all_chunks if c.metadata['profile'] == 'ai_interview']
mixed_chunks = [c for c in all_chunks if c.metadata['profile'] == 'mixed_pdf']

save_jsonl(ai_chunks, OUT_AI / 'all_ai_interview_pdfs_chunks.jsonl')
save_jsonl(mixed_chunks, OUT_MIXED / 'all_different_types_example_chunks.jsonl')

print('Saved per-file and consolidated chunk artifacts.')
print('AI chunks total   :', len(ai_chunks))
print('Mixed chunks total:', len(mixed_chunks))


In [ ]:
# Cell 10: Save quality report + OCR-needed manifest

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
quality_csv = OUT_REPORTS / f'pdf_chunking_quality_report_{run_id}.csv'
quality_json = OUT_REPORTS / f'pdf_chunking_quality_report_{run_id}.json'
ocr_manifest = OUT_REPORTS / f'pdf_ocr_needed_manifest_{run_id}.json'

quality_df.to_csv(quality_csv, index=False)
quality_json.write_text(quality_df.to_json(orient='records', indent=2), encoding='utf-8')

ocr_needed = quality_df[quality_df['ocr_needed'] == True]
ocr_manifest.write_text(ocr_needed.to_json(orient='records', indent=2), encoding='utf-8')

print('Quality report CSV :', quality_csv)
print('Quality report JSON:', quality_json)
print('OCR manifest       :', ocr_manifest)
display(ocr_needed[['file_name', 'profile', 'total_pages', 'nonempty_pages', 'ocr_needed']])


## Notes on `1-2RAG.pdf` (no text layer)

If a PDF has no extractable text layer, text-based chunking cannot proceed reliably.

This notebook flags such files as `ocr_needed=True` in the quality report.

Recommended next step for such files:
1. OCR pipeline (Tesseract/Azure Document Intelligence/AWS Textract)
2. Post-OCR cleaning
3. Same chunking profile + metadata strategy


In [ ]:
# Cell 11: Final summary

print('=' * 80)
print('PDF CHUNKING PIPELINE COMPLETE')
print('=' * 80)
print('Input PDFs                    :', len(all_pdf_files))
print('Total chunks created          :', len(all_chunks))
print('Files flagged OCR needed      :', int(quality_df['ocr_needed'].sum()))
print('Artifacts folder              :', OUT_ROOT)
print('Quality reports folder        :', OUT_REPORTS)
print('\nNotebook complete. You are ready for embedding and vector indexing in next steps.')
